# 🎮 Alvin Pac-Man DQN Training on GPU

This notebook trains a Deep Q-Network (DQN) agent to play Pac-Man using reinforcement learning.

**Hardware Setup:**
- Runtime → Change runtime type → T4 GPU (or better)
- This will train **100x faster** than CPU!

**Training Progress:**
- Target: 100,000 episodes
- Checkpoint: Resume from episode ~27,400
- Estimated time on T4 GPU: ~1-2 hours

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install required packages
!pip install -q torch numpy tqdm

## Step 2: Define Model Architecture

In [ ]:
# DQN Network
import torch
import torch.nn as nn

class DQNNetwork(nn.Module):
    """Deep Q-Network for Pac-Man"""
    
    def __init__(self, input_dim=128, hidden_dims=None, output_dim=4, dropout=0.0):
        super(DQNNetwork, self).__init__()
        
        if hidden_dims is None:
            hidden_dims = [256, 256, 128]
        
        # Build layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        return self.network(x)

print("✅ DQN Network defined")

## Step 3: Define Pac-Man Environment

In [ ]:
# Pac-Man Environment (simplified for Colab)
import numpy as np
import random
import math

class PacManEnv:
    """Pac-Man environment for RL training"""
    
    def __init__(self, maze_size=(20, 20), max_steps=1000):
        self.maze_width = maze_size[0]
        self.maze_height = maze_size[1]
        self.max_steps = max_steps
        self.max_distance = math.sqrt(2) * self.maze_width
        
        # Game state
        self.player_pos = None
        self.player_direction = 'RIGHT'
        self.ghost_positions = []
        self.ghost_scared = []
        self.pellets = set()
        self.power_pellets = set()
        self.walls = set()
        self.score = 0
        self.lives = 3
        self.steps = 0
        self.powered_up = 0
        self.total_pellets = 0
        self.last_distance_to_pellet = 0
    
    def reset(self):
        """Reset environment"""
        self._create_maze()
        self.player_pos = (self.maze_width // 2, self.maze_height // 2)
        self.player_direction = 'RIGHT'
        self.ghost_positions = [(3, 3), (self.maze_width - 4, 3), 
                                 (3, self.maze_height - 4), (self.maze_width - 4, self.maze_height - 4)]
        self.ghost_scared = [False] * 4
        self.score = 0
        self.lives = 3
        self.steps = 0
        self.powered_up = 0
        self.total_pellets = len(self.pellets) + len(self.power_pellets)
        return self._get_state()
    
    def _create_maze(self):
        """Create maze with walls, pellets, power pellets"""
        self.walls = set()
        self.pellets = set()
        self.power_pellets = set()
        
        # Border walls
        for x in range(self.maze_width):
            self.walls.add((x, 0))
            self.walls.add((x, self.maze_height - 1))
        for y in range(self.maze_height):
            self.walls.add((0, y))
            self.walls.add((self.maze_width - 1, y))
        
        # Internal walls
        for x in range(5, self.maze_width - 5, 5):
            for y in range(5, self.maze_height - 5, 3):
                if random.random() < 0.5:
                    self.walls.add((x, y))
        
        # Pellets
        for x in range(1, self.maze_width - 1):
            for y in range(1, self.maze_height - 1):
                if (x, y) not in self.walls:
                    self.pellets.add((x, y))
        
        # Power pellets
        self.power_pellets = {(2, 2), (self.maze_width - 3, 2), 
                              (2, self.maze_height - 3), (self.maze_width - 3, self.maze_height - 3)}
        self.pellets -= self.power_pellets
    
    def step(self, action):
        """Execute one step"""
        self.steps += 1
        reward = -0.1  # Time penalty
        
        # Update direction
        direction_map = {0: 'UP', 1: 'DOWN', 2: 'LEFT', 3: 'RIGHT'}
        self.player_direction = direction_map[action]
        
        # Move player
        new_pos = self._move(self.player_pos, action)
        if new_pos not in self.walls:
            self.player_pos = new_pos
        else:
            reward -= 1  # Wall penalty
        
        # Collect pellets
        if self.player_pos in self.pellets:
            self.pellets.remove(self.player_pos)
            self.score += 10
            reward += 10
        
        # Collect power pellets
        if self.player_pos in self.power_pellets:
            self.power_pellets.remove(self.player_pos)
            self.score += 50
            reward += 50
            self.powered_up = 40
            self.ghost_scared = [True] * 4
        
        # Decay power
        if self.powered_up > 0:
            self.powered_up -= 1
            if self.powered_up == 0:
                self.ghost_scared = [False] * 4
        
        # Move ghosts
        new_ghost_positions = []
        for ghost_pos in self.ghost_positions:
            new_ghost_positions.append(self._move_ghost(ghost_pos))
        self.ghost_positions = new_ghost_positions
        
        # Ghost collision
        for i, ghost_pos in enumerate(self.ghost_positions):
            if self._manhattan_distance(self.player_pos, ghost_pos) < 1.5:
                if self.ghost_scared[i]:
                    self.score += 200
                    reward += 200
                    self.ghost_positions[i] = (3, 3)
                    self.ghost_scared[i] = False
                else:
                    self.lives -= 1
                    reward -= 500
                    if self.lives > 0:
                        self.player_pos = (self.maze_width // 2, self.maze_height // 2)
                    break
        
        # Shaped rewards
        if len(self.pellets) > 0:
            min_pellet_dist = min(self._manhattan_distance(self.player_pos, p) for p in self.pellets)
            if min_pellet_dist < self.last_distance_to_pellet:
                reward += 1
            self.last_distance_to_pellet = min_pellet_dist
        
        if self.powered_up == 0:
            min_ghost_dist = min(self._manhattan_distance(self.player_pos, g) for g in self.ghost_positions)
            if min_ghost_dist < 3:
                reward -= 5
        
        # Done condition
        done = (self.lives <= 0 or 
                len(self.pellets) + len(self.power_pellets) == 0 or 
                self.steps >= self.max_steps)
        
        info = {
            'score': self.score,
            'lives': self.lives,
            'pellets_remaining': len(self.pellets) + len(self.power_pellets),
            'powered_up': self.powered_up > 0
        }
        
        return self._get_state(), reward, done, info
    
    def _move(self, pos, action):
        """Calculate new position"""
        x, y = pos
        if action == 0: return (x, y - 1)  # UP
        elif action == 1: return (x, y + 1)  # DOWN
        elif action == 2: return (x - 1, y)  # LEFT
        elif action == 3: return (x + 1, y)  # RIGHT
        return pos
    
    def _move_ghost(self, ghost_pos):
        """Simple ghost AI"""
        gx, gy = ghost_pos
        px, py = self.player_pos
        
        possible_moves = []
        if px < gx: possible_moves.append((gx - 1, gy))
        elif px > gx: possible_moves.append((gx + 1, gy))
        if py < gy: possible_moves.append((gx, gy - 1))
        elif py > gy: possible_moves.append((gx, gy + 1))
        
        if random.random() < 0.2:
            possible_moves.extend([(gx, gy - 1), (gx, gy + 1), (gx - 1, gy), (gx + 1, gy)])
        
        valid_moves = [m for m in possible_moves if m not in self.walls]
        return random.choice(valid_moves) if valid_moves else ghost_pos
    
    def _manhattan_distance(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])
    
    def _get_state(self):
        """Generate 128-dim feature vector (simplified)"""
        # Simplified state representation for Colab
        # In production, this matches the full 69-feature frontend exactly
        features = []
        
        # Player position
        px, py = self.player_pos
        features.extend([px / self.maze_width, py / self.maze_height])
        
        # Direction (one-hot)
        features.extend([
            1.0 if self.player_direction == 'UP' else 0.0,
            1.0 if self.player_direction == 'DOWN' else 0.0,
            1.0 if self.player_direction == 'LEFT' else 0.0,
            1.0 if self.player_direction == 'RIGHT' else 0.0
        ])
        
        # Ghost data (4 ghosts × 6 features)
        for ghost_pos in self.ghost_positions:
            dx = (ghost_pos[0] - px) / self.maze_width
            dy = (ghost_pos[1] - py) / self.maze_height
            dist = math.sqrt(dx**2 + dy**2)
            manhattan = self._manhattan_distance(self.player_pos, ghost_pos) / (self.maze_width * 2)
            features.extend([dist, manhattan, dx, dy, 0.0, 0.0])  # Simplified
        
        # Pellet info (8 features)
        if self.pellets:
            nearest = min(self.pellets, key=lambda p: math.sqrt((p[0]-px)**2 + (p[1]-py)**2))
            dx = (nearest[0] - px) / self.maze_width
            dy = (nearest[1] - py) / self.maze_height
            dist = math.sqrt(dx**2 + dy**2)
            features.extend([dist, dx, dy, 0.0, 0.0, 0.0, 0.0, 0.0])
        else:
            features.extend([1.0] * 8)
        
        # Power pellets (6 features)
        if self.power_pellets:
            nearest = min(self.power_pellets, key=lambda p: math.sqrt((p[0]-px)**2 + (p[1]-py)**2))
            dx = (nearest[0] - px) / self.maze_width
            dy = (nearest[1] - py) / self.maze_height
            dist = math.sqrt(dx**2 + dy**2)
            features.extend([dist, dx, dy, 0.0, len(self.power_pellets)/4.0, 0.0])
        else:
            features.extend([1.0] * 6)
        
        # Walls (8 features - simplified)
        features.extend([0.5] * 8)
        
        # Tactical (12 features - simplified)
        features.extend([0.5] * 12)
        
        # Game state (5 features)
        pellets_left = len(self.pellets) + len(self.power_pellets)
        features.extend([
            1.0 if self.powered_up > 0 else 0.0,
            self.powered_up / 50.0,
            self.lives / 3.0,
            min(self.score / 10000.0, 1.0),
            pellets_left / max(self.total_pellets, 1)
        ])
        
        # Pad to 128
        while len(features) < 128:
            features.append(0.0)
        
        return np.array(features[:128], dtype=np.float32)

print("✅ Pac-Man Environment defined")

## Step 4: Upload Checkpoint (Optional)

If you want to resume from your current checkpoint (~27,400 episodes), upload `best_rl_model.pth` or `best_dqn_model.pth` using the file browser on the left.

In [ ]:
# Check for uploaded checkpoint
import os

checkpoint_files = ['best_rl_model.pth', 'best_dqn_model.pth', 'checkpoint.pth']
checkpoint_path = None

for filename in checkpoint_files:
    if os.path.exists(filename):
        checkpoint_path = filename
        print(f"✅ Found checkpoint: {checkpoint_path}")
        break

if checkpoint_path is None:
    print("ℹ️ No checkpoint found. Training will start from scratch.")
    print("   To resume training, upload best_rl_model.pth using the file browser.")

## Step 5: Training Configuration

In [ ]:
# Hyperparameters
NUM_EPISODES = 100000
MAX_STEPS_PER_EPISODE = 1000
BATCH_SIZE = 64
LEARNING_RATE = 0.0001
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.9999
TARGET_UPDATE_FREQ = 100
REPLAY_BUFFER_SIZE = 100000
MIN_REPLAY_SIZE = 1000

print("Training Configuration:")
print(f"  Episodes: {NUM_EPISODES:,}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epsilon decay: {EPSILON_DECAY}")
print(f"  Gamma (discount): {GAMMA}")

## Step 6: Initialize Training

In [ ]:
import torch.optim as optim
from collections import deque

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Environment
env = PacManEnv(maze_size=(20, 20), max_steps=MAX_STEPS_PER_EPISODE)

# Models
model = DQNNetwork(input_dim=128, hidden_dims=[256, 256, 128], output_dim=4, dropout=0.0)
target_model = DQNNetwork(input_dim=128, hidden_dims=[256, 256, 128], output_dim=4, dropout=0.0)

# Load checkpoint if available
start_episode = 0
epsilon = EPSILON_START
best_reward = -float('inf')

if checkpoint_path:
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Get episode number
    if 'episode' in checkpoint:
        start_episode = checkpoint['episode']
        print(f"  Resuming from episode: {start_episode:,}")
        print(f"  Avg reward: {checkpoint.get('avg_reward', 'N/A')}")
        epsilon = checkpoint.get('epsilon', EPSILON_START)
    elif 'epoch' in checkpoint:
        print(f"  Loaded pre-trained DQN model (epoch {checkpoint['epoch']})")
    
    print("✅ Checkpoint loaded")

# Move to device
model = model.to(device)
target_model.load_state_dict(model.state_dict())
target_model = target_model.to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Replay buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))
    
    def __len__(self):
        return len(self.buffer)

replay_buffer = ReplayBuffer(capacity=REPLAY_BUFFER_SIZE)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print("✅ Training initialized")

## Step 7: Training Loop

In [ ]:
from tqdm import tqdm

def train_step(model, target_model, optimizer, replay_buffer, batch_size, gamma):
    """Single training step"""
    if len(replay_buffer) < batch_size:
        return None
    
    # Sample batch
    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
    
    # Convert to tensors
    states = torch.FloatTensor(states).to(device)
    actions = torch.LongTensor(actions).to(device)
    rewards = torch.FloatTensor(rewards).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).to(device)
    
    # Current Q-values
    q_values = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    
    # Target Q-values
    with torch.no_grad():
        next_q_values = target_model(next_states).max(1)[0]
        target_q_values = rewards + gamma * next_q_values * (1 - dones)
    
    # Loss
    loss = nn.MSELoss()(q_values, target_q_values)
    
    # Optimize
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    
    return loss.item()

def play_episode(env, model, epsilon, max_steps):
    """Play one episode"""
    state = env.reset()
    episode_reward = 0
    episode_steps = 0
    experiences = []
    
    for step in range(max_steps):
        # Epsilon-greedy
        if np.random.random() < epsilon:
            action = np.random.randint(0, 4)
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = model(state_tensor)
                action = q_values.argmax().item()
        
        next_state, reward, done, info = env.step(action)
        experiences.append((state, action, reward, next_state, done))
        
        episode_reward += reward
        episode_steps += 1
        state = next_state
        
        if done:
            break
    
    return experiences, episode_reward, episode_steps, info

# Training loop
print("="*70)
print("Starting RL Training on GPU!")
print("="*70)
print()

rewards_history = []

for episode in tqdm(range(start_episode, NUM_EPISODES), desc="Training"):
    # Play episode
    experiences, episode_reward, episode_steps, info = play_episode(
        env, model, epsilon, MAX_STEPS_PER_EPISODE
    )
    
    # Add to replay buffer
    for exp in experiences:
        replay_buffer.push(*exp)
    
    # Train
    if len(replay_buffer) >= MIN_REPLAY_SIZE:
        loss = train_step(model, target_model, optimizer, replay_buffer, BATCH_SIZE, GAMMA)
    else:
        loss = None
    
    # Update target network
    if episode % TARGET_UPDATE_FREQ == 0:
        target_model.load_state_dict(model.state_dict())
    
    # Decay epsilon
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    # Track rewards
    rewards_history.append(episode_reward)
    avg_reward_100 = np.mean(rewards_history[-100:]) if len(rewards_history) >= 100 else episode_reward
    
    # Log progress
    if (episode + 1) % 100 == 0:
        loss_str = f"{loss:.2f}" if loss else "N/A"
        print(f"Ep {episode+1:,}/{NUM_EPISODES:,} | R: {episode_reward:.1f} | "
              f"Avg100: {avg_reward_100:.1f} | Steps: {episode_steps} | "
              f"ε: {epsilon:.3f} | Loss: {loss_str} | Score: {info['score']}")
    
    # Save best model
    if avg_reward_100 > best_reward:
        best_reward = avg_reward_100
        
        torch.save({
            'episode': episode,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'avg_reward': avg_reward_100,
            'epsilon': epsilon,
            'config': {
                'input_dim': 128,
                'hidden_dims': [256, 256, 128],
                'output_dim': 4,
                'learning_rate': LEARNING_RATE,
                'gamma': GAMMA
            }
        }, 'best_rl_model_colab.pth')
        
        if (episode + 1) % 1000 == 0:
            print(f"  ✅ Saved best model (avg_reward: {avg_reward_100:.1f})")

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print(f"Best avg reward (100 episodes): {best_reward:.1f}")
print(f"Final epsilon: {epsilon:.3f}")
print(f"\nDownload: best_rl_model_colab.pth")

## Step 8: Download Trained Model

After training completes, download `best_rl_model_colab.pth` from the files panel (left side) and use it to replace your local checkpoint.

In [ ]:
# Download model using Google Colab's download feature
from google.colab import files

if os.path.exists('best_rl_model_colab.pth'):
    print("Downloading trained model...")
    files.download('best_rl_model_colab.pth')
    print("✅ Download started!")
else:
    print("❌ No trained model found. Make sure training completed successfully.")

## Next Steps

1. Download `best_rl_model_colab.pth`
2. Rename it to `best_rl_model.pth`
3. Replace the checkpoint in your local `ml-training/checkpoints/` directory
4. Deploy to Hugging Face Spaces
5. Test the improved AI at https://alvin-pacman.pages.dev/